In [ ]:
!pip install edgartools earningscall


In [ ]:
!pip install notebook-intelligence

In [ ]:
!pip install libsql

In [37]:
import libsql

In [2]:
import numpy as np
import pandas as pd
import yfinance as yf
from edgar import *
from earningscall import get_company
from datetime import datetime,timedelta
import re
from libsql_client import create_client_sync

In [117]:
import libsql
import os
from dotenv import load_dotenv

load_dotenv()

try:
    # Testing the native driver
    conn = libsql.connect(database=os.getenv("TURSO_URL"), auth_token=os.getenv("TURSO_TOKEN"))
    cursor = conn.cursor()
except Exception as e:
    print(f"❌ Environment Error: {e}")

In [ ]:
client.execute('''
    CREATE TABLE IF NOT EXISTS sec_chunks (
        chunk_id TEXT PRIMARY KEY,
        doc_id TEXT,
        ticker TEXT,
        filing_date TEXT,
        item_type TEXT,
        chunk_index INTEGER,
        content TEXT,
        sentiment_score REAL
    )
''')
print("Table 'sec_chunks' is live on Turso.")

In [45]:
set_identity(os.getenv("SEC_API_USER_AGENT"))

In [ ]:
import re



In [ ]:
filings = Company("MSFT").get_filings(form="10-K").latest()
xbrl = filings.xbrl()
statements = xbrl.statements

# Display financial statements
balance_sheet = statements.balance_sheet()
income_statement = statements.income_statement()
cash_flow = statements.cashflow_statement()


In [ ]:
statements

In [ ]:
income_df = income_statement.to_dataframe()

In [ ]:
income_df

In [ ]:
cash_flow = cash_flow.to_dataframe()


In [ ]:
cash_flow

In [ ]:
# balance_sheet = balance_sheet.to_dataframe()
pd.set_option('display.max_rows', 110)  
balance_sheet

In [74]:
company = Company("MSFT")
    # We go back to 2018 to get a full 8 years of historical context
filings = company.get_filings(form="10-Q").latest()

In [99]:
# r = filings.obj()
print(r.filing_date)
fin = r.financials.cash_flow_statement().to_dataframe()
pd.set_option('display.max_rows', None, 'display.max_columns', None)
fin[fin.standard_concept=='NetCashFromOperatingActivities']


2026-01-28


,concept,label,standard_concept,2025-12-31 (Q2),2024-12-31 (Q2),2025-12-31 (YTD),2024-12-31 (YTD),level,abstract,dimension,is_breakdown,dimension_axis,dimension_member,dimension_member_label,dimension_label,balance,weight,preferred_sign,parent_concept,parent_abstract_concept
12,us-gaap_IncreaseDecreaseInOtherNoncurrentAssets,Other long-term assets,NetCashFromOperatingActivities,-1.288000e+09,-1.089000e+09,-1.682000e+09,-2.850000e+09,3,False,False,False,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,us-gaap_NetCashProvidedByUsedInOperatingActivi...,us-gaap_IncreaseDecreaseInOperatingCapitalAbst...
17,us-gaap_IncreaseDecreaseInOtherNoncurrentLiabi...,Other long-term liabilities,NetCashFromOperatingActivities,-1.193000e+09,7.160000e+08,-1.670000e+09,6.830000e+08,3,False,False,False,NaN,NaN,NaN,NaN,NaN,1.0,1.0,us-gaap_NetCashProvidedByUsedInOperatingActivi...,us-gaap_IncreaseDecreaseInOperatingCapitalAbst...
18,us-gaap_NetCashProvidedByUsedInOperatingActivi...,Net cash from operations,NetCashFromOperatingActivities,3.575800e+10,2.229100e+10,8.081500e+10,5.647100e+10,2,False,False,False,NaN,NaN,NaN,NaN,debit,1.0,1.0,us-gaap_CashCashEquivalentsRestrictedCashAndRe...,us-gaap_NetCashProvidedByUsedInOperatingActivi...


In [ ]:
def isolate_q4_safely(raw_df):
    """
    Takes the raw SEC DataFrame and cleanly isolates Q4 for complete years,
    while safely passing through incomplete years (like 2026).
    """
    final_records = []
    
    # Process sequentially by year
    for year in sorted(raw_df['fiscal_year'].unique()):
        year_data = raw_df[raw_df['fiscal_year'] == year]
        
        # 1. Isolate the pieces
        qs = year_data[year_data['fiscal_period'].isin(['Q1', 'Q2', 'Q3'])]
        fy = year_data[year_data['fiscal_period'] == 'FY']
        
        # 2. Always append the standard quarters we have
        for _, row in qs.iterrows():
            final_records.append(row.to_dict())
            
        # 3. The Strict Guardrail for Q4
        # We only calculate Q4 if we have the FY filing AND we have exactly 3 prior quarters.
        if not fy.empty:
            if len(qs) == 3:
                fy_row = fy.iloc[0].copy()
                
                # Flow Metrics Subtraction
                fy_row['revenue'] = fy_row['revenue'] - qs['revenue'].sum()
                fy_row['net_income'] = fy_row['net_income'] - qs['net_income'].sum()
                fy_row['op_cash_flow'] = fy_row['op_cash_flow'] - qs['op_cash_flow'].sum()
                fy_row['capex'] = fy_row['capex'] - qs['capex'].sum()
                
                # Snapshot Metrics (cash_eq, total_liability, etc.) remain untouched
                
                # Update Identifiers
                fy_row['fiscal_period'] = 'Q4'
                fy_row['filing_id'] = fy_row['filing_id'].replace('FY', 'Q4')
                
                final_records.append(fy_row.to_dict())
            else:
                print(f" Alert: FY {year} is missing a Q1/Q2/Q3 filing. Cannot safely calculate Q4.")
        else:
            # This handles 2026 perfectly - it silently ignores Q4 and moves on
            print(f"ℹ Info: FY {year} is incomplete (No 10-K yet). Skipping Q4 calculation.")

    # Convert back to DataFrame and sort chronologically
    clean_df = pd.DataFrame(final_records)
    clean_df = clean_df.sort_values(['fiscal_year', 'fiscal_period'], ascending=[False, False]).reset_index(drop=True)
    
    return clean_df

# Execute the transformation
# final_financials_df = isolate_q4_safely(data)

In [110]:
data.tail(10)

,filing_id,ticker,filing_date,effective_date,fiscal_year,fiscal_period,revenue,net_income,op_cash_flow,capex,cash_eq,total_liability,total_assets,total_equity
19,MSFT_2021_Q3_FILING,MSFT,2021-04-27,2021-04-28,2021,Q3,4.170600e+10,1.545700e+10,2.217900e+10,-5.089000e+09,1.370200e+10,1.743740e+11,3.088790e+11,1.345050e+11
20,MSFT_2021_Q2_FILING,MSFT,2021-01-26,2021-01-27,2021,Q2,4.307600e+10,1.546300e+10,1.251600e+10,-4.174000e+09,1.443200e+10,1.739010e+11,3.041370e+11,1.302360e+11
21,MSFT_2021_Q1_FILING,MSFT,2020-10-27,2020-10-28,2021,Q1,3.715400e+10,1.389300e+10,1.933500e+10,-4.907000e+09,1.720500e+10,1.776090e+11,3.010010e+11,1.233920e+11
22,MSFT_2020_FY_FILING,MSFT,2020-07-30,2020-07-31,2020,FY,1.430150e+11,4.428100e+10,6.067500e+10,-1.544100e+10,1.357600e+10,1.830070e+11,3.013110e+11,1.183040e+11
23,MSFT_2020_Q3_FILING,MSFT,2020-04-29,2020-04-30,2020,Q3,3.502100e+10,1.075200e+10,1.750400e+10,-3.767000e+09,1.171000e+10,1.709480e+11,2.854490e+11,1.145010e+11
24,MSFT_2020_Q2_FILING,MSFT,2020-01-29,2020-01-30,2020,Q2,3.690600e+10,1.164900e+10,1.068000e+10,-3.545000e+09,8.864000e+09,1.726850e+11,2.827940e+11,1.101090e+11
25,MSFT_2020_Q1_FILING,MSFT,2019-10-23,2019-10-24,2020,Q1,3.305500e+10,1.067800e+10,1.381800e+10,-3.385000e+09,1.311700e+10,1.728940e+11,2.789550e+11,1.060610e+11
26,MSFT_2019_FY_FILING,MSFT,2019-08-01,2019-08-02,2019,FY,1.258430e+11,3.924000e+10,5.218500e+10,-1.392500e+10,1.135600e+10,1.842260e+11,2.865560e+11,1.023300e+11
27,MSFT_2019_Q3_FILING,MSFT,2019-04-24,2019-04-25,2019,Q3,3.057100e+10,8.809000e+09,1.352000e+10,-2.565000e+09,1.121200e+10,1.684170e+11,2.632810e+11,9.486400e+10
28,MSFT_2019_Q2_FILING,MSFT,2019-01-30,2019-01-31,2019,Q2,3.247100e+10,8.420000e+09,8.900000e+09,-3.707000e+09,6.638000e+09,1.667310e+11,2.588590e+11,9.212800e+10


In [ ]:

def map_msft_fiscal_metadata(filing_date, form_type):
    """
    filing_date_str: 'YYYY-MM-DD'
    form_type: '10-K' or '10-Q'
    
    Returns: (fiscal_year, fiscal_period)
    """
    month = filing_date.month
    year = filing_date.year
    
    if "10-K" in form_type:
        # 10-K is filed after June 30, belongs to that calendar year
        return year, "FY"
    
    elif "10-Q" in form_type:
        # Microsoft Q1 is filed in Oct/Nov (belongs to NEXT fiscal year)
        if month in [10, 11, 12]:
            return year + 1, "Q1"
        # Microsoft Q2 is filed in Jan/Feb
        elif month in [1, 2, 3]:
            return year, "Q2"
        # Microsoft Q3 is filed in Apr/May
        elif month in [4, 5, 6]:
            return year, "Q3"
            
    return None, None

# Example Usage:
# f_year, f_period = map_msft_fiscal_metadata("2025-10-23", "10-Q")
# Result: (2026, "Q1")

def get_core_financials(ticker="MSFT"):
    """
    Extracts financial metrics from XBRL statements.
    """
    company = Company(ticker)
    # We go back to 2018 to get a full 8 years of historical context
    filings = company.get_filings(form=["10-K", "10-Q"]).filter(filing_date="2019-01-01:")
    raw_data = []
    for filing in filings:
        print(f"Processing {filing.form} filed on {filing.filing_date}...")
        f_year, f_period = map_msft_fiscal_metadata(filing.filing_date, filing.form)            
        f_id = f"{ticker}_{f_year}_{f_period}_FILING"
        effective_date = (filing.filing_date + timedelta(days=1)).strftime('%Y-%m-%d')

        report = filing.obj()

        # 1. Income Statement Data
        income_df = report.financials.income_statement().to_dataframe()
        # We look for the most common XBRL tags (Concepts)
        revenue = income_df[income_df['standard_concept']=='Revenue'].iloc[0, 3]
        net_income = income_df[income_df['standard_concept']=='NetIncome'].iloc[0, 3]

        # 2. Cash Flow Data
        cf_df = report.financials.cashflow_statement().to_dataframe()
        op_cash_flow = cf_df[cf_df['label']=='Net cash from operations'].iloc[0, 3]
        capex = cf_df[cf_df['standard_concept']=='CapitalExpenses'].iloc[0, 3]


        #3. Balance sheet
        bal_df = report.financials.balance_sheet().to_dataframe()
        cash_eq = bal_df[bal_df['standard_concept']=='CashAndMarketableSecurities'].iloc[0, 3]
        tot_assets = bal_df[bal_df['standard_concept']=='Assets'].iloc[0, 3]
        tot_liab = bal_df[bal_df['standard_concept']=='Liabilities'].iloc[0, 3]
        tot_eq = bal_df[bal_df['standard_concept']=='AllEquityBalance'].iloc[0, 3]
        
        raw_data.append({
            'filing_id': f_id,
            'ticker': ticker,
            'filing_date': filing.filing_date,
            'effective_date': effective_date, 
            'fiscal_year': f_year,
            'fiscal_period': f_period,
            "revenue": float(revenue),
            "net_income": float(net_income),
            "op_cash_flow": float(op_cash_flow),
            "capex":float(capex),
            "cash_eq":float(cash_eq),
            "total_liability":float(tot_liab),
            "total_assets":float(tot_assets),
            "total_equity":float(tot_eq)}
        )
    return pd.DataFrame(raw_data)     

data = get_core_financials()

Processing 10-Q filed on 2026-01-28...
Processing 10-Q filed on 2025-10-29...
Processing 10-K filed on 2025-07-30...
Processing 10-Q filed on 2025-04-30...
Processing 10-Q filed on 2025-01-29...
Processing 10-Q filed on 2024-10-30...
Processing 10-K filed on 2024-07-30...
Processing 10-Q filed on 2024-04-25...
Processing 10-Q filed on 2024-01-30...
Processing 10-Q filed on 2023-10-24...
Processing 10-K filed on 2023-07-27...
Processing 10-Q filed on 2023-04-25...
Processing 10-Q filed on 2023-01-24...
Processing 10-Q filed on 2022-10-25...
Processing 10-K filed on 2022-07-28...
Processing 10-Q filed on 2022-04-26...
Processing 10-Q filed on 2022-01-25...
Processing 10-Q filed on 2021-10-26...
Processing 10-K filed on 2021-07-29...
Processing 10-Q filed on 2021-04-27...
Processing 10-Q filed on 2021-01-26...
Processing 10-Q filed on 2020-10-27...
Processing 10-K filed on 2020-07-30...
Processing 10-Q filed on 2020-04-29...
Processing 10-Q filed on 2020-01-29...
Processing 10-Q filed on 

In [114]:
def isolate_q4_safely(raw_df):
    """
    Takes the raw SEC DataFrame and cleanly isolates Q4 for complete years,
    while safely passing through incomplete years (like 2026).
    """
    final_records = []
    
    # Process sequentially by year
    for year in sorted(raw_df['fiscal_year'].unique()):
        year_data = raw_df[raw_df['fiscal_year'] == year]
        
        # 1. Isolate the pieces
        qs = year_data[year_data['fiscal_period'].isin(['Q1', 'Q2', 'Q3'])]
        fy = year_data[year_data['fiscal_period'] == 'FY']
        
        # 2. Always append the standard quarters we have
        for _, row in qs.iterrows():
            final_records.append(row.to_dict())
            
        # 3. The Strict Guardrail for Q4
        # We only calculate Q4 if we have the FY filing AND we have exactly 3 prior quarters.
        if not fy.empty:
            if len(qs) == 3:
                fy_row = fy.iloc[0].copy()
                
                # Flow Metrics Subtraction
                fy_row['revenue'] = fy_row['revenue'] - qs['revenue'].sum()
                fy_row['net_income'] = fy_row['net_income'] - qs['net_income'].sum()
                fy_row['op_cash_flow'] = fy_row['op_cash_flow'] - qs['op_cash_flow'].sum()
                fy_row['capex'] = fy_row['capex'] - qs['capex'].sum()
                
                # Snapshot Metrics (cash_eq, total_liability, etc.) remain untouched
                
                # Update Identifiers
                fy_row['fiscal_period'] = 'Q4'
                fy_row['filing_id'] = fy_row['filing_id'].replace('FY', 'Q4')
                
                final_records.append(fy_row.to_dict())
            else:
                print(f" Alert: FY {year} is missing a Q1/Q2/Q3 filing. Cannot safely calculate Q4.")
        else:
            # This handles 2026 perfectly - it silently ignores Q4 and moves on
            print(f"ℹ Info: FY {year} is incomplete (No 10-K yet). Skipping Q4 calculation.")

    # Convert back to DataFrame and sort chronologically
    clean_df = pd.DataFrame(final_records)
    clean_df['filing_date'] = clean_df['filing_date'].astype(str)
    clean_df['effective_date'] = clean_df['effective_date'].astype(str)
    clean_df = clean_df.sort_values(['fiscal_year', 'fiscal_period'], ascending=[False, False]).reset_index(drop=True)
    
    return clean_df

# Execute the transformation
# final_financials_df = isolate_q4_safely(data)

In [115]:
final = isolate_q4_safely(data)

 Alert: FY 2019 is missing a Q1/Q2/Q3 filing. Cannot safely calculate Q4.
ℹ Info: FY 2026 is incomplete (No 10-K yet). Skipping Q4 calculation.


In [118]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS financial_filings_raw (
    filing_id TEXT PRIMARY KEY,
    ticker TEXT,
    filing_date TEXT,
    effective_date TEXT,
    fiscal_year INTEGER,
    fiscal_period TEXT,
    revenue REAL,
    net_income REAL,
    op_cash_flow REAL,
    capex REAL,
    cash_eq REAL,
    total_liability REAL,
    total_assets REAL,
    total_equity REAL
)
""")
conn.commit()

records_to_insert = final[[
    'filing_id', 'ticker', 'filing_date', 'effective_date', 
    'fiscal_year', 'fiscal_period', 'revenue', 'net_income', 
    'op_cash_flow', 'capex', 'cash_eq', 'total_liability', 
    'total_assets', 'total_equity'
]].values.tolist()

# Execute batch insert
cursor.executemany("""
    INSERT OR REPLACE INTO financial_filings_raw 
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", records_to_insert)

conn.commit()
print(f"✅ Successfully staged {len(records_to_insert)} financial quarters into Turso.")

✅ Successfully staged 28 financial quarters into Turso.


In [ ]:
tenk = filings.obj().document
item_1a = tenk.get_section("Item 2")
if item_1a:
    # Note: Section objects also have text() method
    text = item_1a.text(include_tables=False,table_max_col_width=None)
    print(text)

In [ ]:
for f in filings:
    print(f.filing_date.isoformat().replace('-','_'))

# eightk = filings[0].obj()
# print(eightk['Item 1'])
# Get basic information
# report_date = eightk.date_of_report
# items_reported = eightk.items
# print(items_reported)

In [ ]:
'2023-3-3'.replace('-','')

In [ ]:
def clean_prose(text):
    """
    Cleans SEC prose for NLP: Normalizes whitespace and removes boilerplate headers.
    """
    if not text: return ""
    # Normalize whitespace (replaces tabs/newlines with single space)
    text = re.sub(r'\s+', ' ', text)
    # Remove standard SEC 'Table of Contents' and page markers
    text = re.sub(r'(Table of Contents|Item [0-9A-Z]+\.)', '', text, flags=re.IGNORECASE)
    return text.strip()

def get_10k_data(ticker='MSFT',years=5 ):
    company = Company('MSFT')
        # Get the last 5 10-Ks (filing once per year)
    ten_ks = company.get_filings(form="10-K").latest(years)
    data =[]
    for filing in ten_ks:
        
        f_date = filing.filing_date.isoformat()
        doc_id = f"{ticker}_10K_{f_date.replace('-','_')}"
        
        print(f"Processing 10-K filed on: {f_date}")
    
        doc = filing.obj()
        
        # Item 1: Business (Dictionary for RAG)
        raw_biz = doc['Item 1']
        clean_biz = clean_prose(raw_biz) if raw_biz else ""
    
        # Item 1A: Risk Factors (Alpha for NLP)
        raw_risk = doc["Item 1A"]
        clean_risk = clean_prose(raw_risk) if raw_risk else ""
        
        data.append({'doc_id':doc_id,'filing_date':f_date,'item1_biz':clean_biz,'item1a_risk':clean_risk})
        # Atomic Insert: Ensures 10-K data is linked to the date the market saw it
         
    print("10-K Ingestion Complete.")
    return data

In [ ]:
data = get_10k_data()

In [ ]:
def get_10q_data():
    

In [ ]:
def semantic_sentence_splitter(text, max_chars=1500):
    """Splits 'wall of text' into chunks at sentence boundaries (~350-400 tokens)."""
    if not text: return []
    # Regex split: period/exclamation/question followed by space
    sentences = re.split(r'(?<=[.!?]) +', text)
    chunks, current_chunk = [], ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) > max_chars and current_chunk:
            chunks.append(current_chunk.strip())
            current_chunk = sentence
        else:
            current_chunk += " " + sentence
    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks

In [ ]:
def sync_10k_to_turso(data_list, ticker="MSFT"):
    """
    data_list: [{'doc_id': '...', 'item1_biz': '...', 'item1a_risk': '...'}]
    """
    statements = []

    for entry in data_list:
        doc_id = entry['doc_id']
        
        filing_date = entry['filing_date']

        # Define which blobs to process
        sections = [
            ('Item 1', entry['item1_biz']),
            ('Item 1A', entry['item1a_risk'])
        ]

        for item_label, blob in sections:
            chunks = semantic_sentence_splitter(blob)
            
            for i, content in enumerate(chunks):
                # Unique PK: MSFT_20260421_ITEM1A_0
                chunk_id = f"{ticker}_{filing_date}_{item_label.replace(' ', '')}_{i}"
                
                # The SQL statement and its parameters
                sql = """
                INSERT OR REPLACE INTO sec_chunks 
                (chunk_id, doc_id, ticker, filing_date, item_type, chunk_index, content) 
                VALUES (?, ?, ?, ?, ?, ?, ?)
                """
                params = (chunk_id, doc_id, ticker, filing_date, item_label, i, content)
                statements.append((sql, params))

    # 2. Batch upload to Turso (Very efficient over the network)
    if statements:
        print(f"Preparing to sync {len(statements)} chunks to Turso...")
        print("Sample params:", statements[0][1])  
        client.batch(statements)
        print("Done! Data is now in the cloud.")
        
    else:
        print("No data found to sync.")

In [ ]:
sync_10k_to_turso(data)

In [120]:
filing = Company("MSFT").get_filings(form="10-K").latest()
r = filing.obj()


TenK falling back to legacy parser for 'Item7' (filing: 0000950170-25-100235). New parser sections available: ['Item 5', 'Item 10', 'Item 11', 'Item 12', 'Item 13', 'Item 14', 'Item 15', 'Item 16', 'Item 1', 'Item 1A', 'Item 1B', 'Item 1C', 'Item 2', 'Item 3', 'Item 4', 'Item 6', 'Item 7', 'Item 7A', 'Item 8', 'Item 9', 'Item 9A', 'Item 9B', 'Item 9C']. This fallback will be removed in v6.0.


None


In [122]:
with open("./10_K.txt", "w") as file:
    file.write(r['Item 7'])

print()

In [ ]:
r.

"ITEM 1A. RISK FACTORS\n\nOur operations and financial results are subject to various risks and uncertainties, including those described below, that could adversely affect our business, operations, financial condition, results of operations, liquidity, and the trading price of our common stock.\n\nSTRATEGIC AND COMPETITIVE RISKS\n\nWe face intense competition across all markets for our products and services, which could adversely affect our results of operations.\n\nCompetition in the technology sector\n\nOur competitors range in size from diversified global companies with significant research and development resources to small, specialized firms whose narrower product lines may let them be more effective in deploying technical, marketing, and financial resources. Barriers to entry in many of our businesses are low and many of the areas in which we compete evolve rapidly with changing and disruptive technologies, shifting user needs, and frequent introductions of new products and servi

In [141]:
import re

import re

def clean_mda_narrative(raw_text):
    if not raw_text: return ""
    text = raw_text.replace('\r\n', '\n')
    paragraphs = re.split(r'\n{2,}', text)
    clean_paragraphs = []

    # Expanded blacklist for common 10-K/10-Q artifacts
    debris_blacklist = {
        'three months ended', 'six months ended', 'nine months ended', 'year ended',
        'percentage change', 'percentage', 'change', 'in millions', 'in billions',
        'unaudited', 'consolidated statements', 'revenue', 'gross margin',
        'operating income', 'total', 'total revenue', 'december 31,', 'march 31,', 
        'june 30,', 'september 30,', 'as a percent of revenue', 'percentagechange',
        'net income', 'operating expenses', 'cost of revenue'
    }

    for p in paragraphs:
        lines = p.split('\n')
        surviving_lines = []

        for line in lines:
            line_str = line.strip()
            if not line_str: continue
            
            # 1. Blacklist Check
            if line_str.lower() in debris_blacklist: continue

            # 2. Footnote & Boilerplate Check
            if line_str.startswith('(a)') or line_str.startswith('(b)') or 'Refer to Note' in line_str:
                continue
                
            # 3. The "Ghost Table" Word Count Check (KILLS ROW HEADERS)
            # A real sentence has more than 4 words. 
            words = line_str.split()
            if len(words) < 5:
                continue

            # 4. The Density Test (KILLS NUMBERS/SYMBOLS)
            total_chars = len(line_str)
            alpha_chars = sum(c.isalpha() for c in line_str)
            if total_chars > 0 and (alpha_chars / total_chars) < 0.5:
                continue 

            surviving_lines.append(line_str)

        if surviving_lines:
            clean_paragraphs.append(" ".join(surviving_lines))

    return "\n\n".join(clean_paragraphs)


def semantic_sentence_splitter(text, max_chars=1200):
    """Splits text into RAG-friendly chunks at sentence boundaries."""
    if not text: return []
    sentences = re.split(r'(?<=[.!?]) +', text)
    chunks, current_chunk = [], ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) > max_chars and current_chunk:
            chunks.append(current_chunk.strip())
            current_chunk = sentence
        else:
            current_chunk += " " + sentence
    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks

In [151]:
def clean_mda_narrative(raw_text):
    """
    Cleans SEC MD&A text by evaporating fragmented table rows using 
    alphabetic density and word-count thresholds, preserving narrative context.
    """
    if not raw_text: return ""

    # 1. Standardize line breaks and split
    lines = raw_text.replace('\r\n', '\n').split('\n')
    clean_lines = []

    # 2. Blacklist of orphaned table row headers and SEC boilerplate
    debris_blacklist = {
        'three months ended', 'six months ended', 'nine months ended', 'year ended',
        'percentage change', 'in millions', 'in billions', 'unaudited',
        'consolidated statements', 'total revenue', 'gross margin',
        'operating income', 'net income', 'operating expenses', 'cost of revenue',
        'as a percent of revenue', 'percentagechange', 'diluted earnings per share'
    }

    # Regex to catch dynamic headers like "Fiscal Year 2025 Compared with 2024"
    comparison_header_regex = r'(?:Fiscal Year|Three Months Ended|Six Months Ended).*?Compared with'

    for line in lines:
        line_str = line.strip()
        
        # Preserve natural paragraph spacing
        if not line_str:
            clean_lines.append("")
            continue

        # Filter A: Is it known table debris?
        if line_str.lower() in debris_blacklist: 
            continue
            
        # Filter B: Is it a repeating comparative header?
        if re.search(comparison_header_regex, line_str, re.IGNORECASE):
            continue

        # Filter C: The "Table Cell" Density Test
        # Real sentences are mostly letters. Table cells are mostly numbers/symbols.
        total_chars = len(line_str)
        alpha_chars = sum(c.isalpha() for c in line_str)
        if total_chars > 0 and (alpha_chars / total_chars) < 0.5:
            continue # Drops lines like "$ 81,273" or "17%"

        # Filter D: The "Ghost Table Row" Check
        # Executives don't write 3-word sentences in an MD&A. 
        # If it's short, it's an orphaned table row label.
        if len(line_str.split()) < 5:
            continue

        # If it survives all filters, it is high-value narrative.
        clean_lines.append(line_str)

    # Re-join into a document
    text = "\n".join(clean_lines)
    
    # Snap the massive gaps (where tables used to be) back into clean double-newlines
    text = re.sub(r'\n{3,}', '\n\n', text).strip()

    return text

In [144]:
with open("./10_K.txt", "w") as file:
    file.write(clean_mda_narrative(r['Item 7']))


In [148]:
with open('./10_K_raw.text','w') as f:
          f.write(r.management_discussion)

In [152]:
def isolate_alpha_chapters(raw_mda_text):
    """
    Ruthlessly slices the MD&A, keeping only the high-value volatility drivers
    and dropping the glossary, taxes, and accounting boilerplate.
    """
    if not raw_mda_text:
        return ""
        
    # The sections that actually move the stock price
    target_headers = [
        r'Economic Conditions, Challenges, and Risks',
        r'Fiscal Year \d{4} Compared with Fiscal Year \d{4}',
        r'Reportable Segments',
        r'Other Planned Uses of Capital'
    ]
    
    # The sections that waste model compute
    stop_headers = [
        r'Metrics',                   # Kills the glossary
        r'OTHER INCOME \(EXPENSE\)',  # Kills derivative hedging talk
        r'INCOME TAXES',              # Kills IRS/tax legislation talk
        r'CRITICAL ACCOUNTING ESTIMATES', # Kills accounting theory
        r'STATEMENT OF MANAGEMENT'    # Kills signatures
    ]
    
    target_regex = re.compile("|".join(target_headers), re.IGNORECASE)
    stop_regex = re.compile("|".join(stop_headers), re.IGNORECASE)
    
    lines = raw_mda_text.replace('\r\n', '\n').split('\n')
    extracted_chapters = []
    is_reading = False
    
    for line in lines:
        line_str = line.strip()
        if not line_str:
            continue
            
        # Toggle ON if we hit a high-value header
        if target_regex.search(line_str):
            is_reading = True
            
        # Toggle OFF if we hit regulatory noise
        if stop_regex.search(line_str):
            is_reading = False
            
        if is_reading:
            extracted_chapters.append(line_str)
            
    return "\n".join(extracted_chapters)

In [160]:
routed=isolate_alpha_chapters(r.management_discussion)
clean = clean_mda_narrative(routed)
with open("./10_K_clean.txt", "w") as file:
    file.write(clean)

# filings.obj()['Item 2']

In [ ]:
print(clean_mda_narrative(r['Item 7']))

In [159]:
clean

"•Microsoft Cloud revenue increased 23% to $168.9 billion.\n•Microsoft 365 Commercial products and cloud services revenue increased 14% driven by Microsoft 365 Commercial cloud revenue growth of 15%.\n•Microsoft 365 Consumer products and cloud services revenue increased 11% driven by Microsoft 365 Consumer cloud revenue growth of 11%.\n•Dynamics products and cloud services revenue increased 15% driven by Dynamics 365 revenue growth of 19%.\n•Server products and cloud services revenue increased 23% driven by Azure and other cloud services revenue growth of 34%.\n•Windows OEM and Devices revenue increased 3%.\n•Xbox content and services revenue increased 16%.\n•Search and news advertising revenue excluding traffic acquisition costs increased 20%.\nOur industry is dynamic and highly competitive, with frequent changes in both technologies and business models. Each industry shift is an opportunity to conceive new products, new technologies, or new ideas that can further transform the indust

In [ ]:
modified_text = clean_mda_narrative(text)

In [ ]:
modified_text

In [ ]:
filings.obj().reports

In [ ]:
with open("./10_Q.txt", "w") as file:
    file.write(filings.obj()['Item 2'])

# filings.obj()['Item 2']

In [ ]:
for f in filings:
    print(f.parse())
    
    print(f.filing_date, f.form)

In [ ]:
sections = filings.sections()

In [ ]:
sections

In [ ]:


# Required for SEC EDGAR access


def clean_tabular_noise(text):
    """Strips lines that are likely dense tables or numeric noise."""
    if not text: return ""
    lines = str(text).split('\n')
    # Filter out lines that have 4 or more numeric/financial symbols
    cleaned_lines = [line for line in lines if len(re.findall(r'\d|\$|%', line)) < 4]
    return " ".join(cleaned_lines).strip()

def ingest_sec_data(ticker="MSFT", years=5):
    """Fetches and cleans 10-K, 10-Q, and 8-K filings."""
    end_date = datetime(2026, 4, 20)
    start_date = end_date - timedelta(days=years*365)
    date_range = f"{start_date.strftime('%Y-%m-%d')}:{end_date.strftime('%Y-%m-%d')}"
    
    company = Company(ticker)
    # Fetching 8-K is crucial for real-time volatility triggers
    filings = company.get_filings(form=["10-K", "10-Q", "8-K"], filing_date=date_range)
    
    processed_docs = []
    for f in filings:
        try:
            doc = f.obj()
            entry = {"date": f.filing_date, "form": f.form, "accession": f.accession_number}
            
            # Using bracket notation to avoid AttributeError in TenQ
            if f.form == "10-K":
                entry["mda"] = doc['Item 7']
                entry["risk_factors"] = doc['Item 1A']
                entry["business_info"] = doc['Item 1'] # Future RAG context
            elif f.form == "10-Q":
                entry["mda"] = doc['Part I, Item 2'] # Fixed key for TenQ
                entry["risk_factors"] = doc['Part II, Item 1A']
            elif f.form == "8-K":
                entry["mda"] = f.text() # 8-Ks are often unstructured press releases
            
            # Apply tabular cleaning immediately
            entry["mda_clean"] = clean_tabular_noise(entry.get("mda", ""))
            entry["risk_clean"] = clean_tabular_noise(entry.get("risk_factors", ""))
            
            processed_docs.append(entry)
        except Exception as e:
            print(f"Error parsing {f.accession_number} ({f.form}): {e}")
            
    return pd.DataFrame(processed_docs)

def ingest_market_data(ticker="MSFT", years=5):
    """Fetches daily price data and the 10-Year Treasury yield."""
    tickers = [ticker, "^TNX", "^VIX"] # MSFT, 10Y Yield, and Volatility Index
    data = yf.download(tickers, start="2021-01-01", end="2026-04-20")
    
    # Flatten multi-index columns for easier SQL ingestion
    data.columns = ['_'.join(col).strip() for col in data.columns.values]
    return data.reset_index()

In [ ]:
sec_data  = ingest_sec_data(years=1)

In [40]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS transcripts (
    transcript_id TEXT PRIMARY KEY,
    ticker TEXT,
    event_timestamp TEXT,
    effective_date TEXT,
    fiscal_year INTEGER,
    fiscal_period TEXT,
    content_prepped TEXT,
    content_qa TEXT,
    sentiment_score REAL DEFAULT 0.0
)
""")

In [42]:
import earningscall
from datetime import datetime, timedelta
import json
# --- HARD CONSTANTS ---
START_DATE = datetime(2019, 1, 1)
END_DATE = datetime(2026, 4, 22) # Current Date
def process_msft_history(cursor, ticker="MSFT"):
    company = earningscall.get_company(ticker)
    
    # 1. Define the 5-year boundary
    
    # Storage for all processed quarters
    historical_data = []

    # 2. Iterate through events
    for event in company.events():
        # Skip future events
        event_date = event.conference_date.replace(tzinfo=None)
        if not (START_DATE <= event_date <= END_DATE):
            continue

        # 2. EFFECTIVE MARKET DATE (T+1 Logic)
        # If call is 4:00 PM (16:00) or later, the market reacts the NEXT day
        # if event.conference_date.hour >= 16:
        effective_date = (event.conference_date + timedelta(days=1)).strftime('%Y-%m-%d')
        # else:
            # effective_date = event.conference_date.strftime('%Y-%m-%d')
        print(f"Ingesting: Q{event.quarter} {event.year} (Date: {event.conference_date.strftime('%Y-%m-%d')})")
        
        # 3. FISCAL ALIGNMENT
        # We use the library's year/quarter as the "Source of Truth" for labeling
        f_year = event.year
        f_period = f"Q{event.quarter}" if event.quarter < 4 else "FY"
        t_id = f"{ticker}_{f_year}_{f_period}_TRANSCRIPT"
        
        try:
            # Fetch Level 2 to get speaker names and text
            transcript = company.get_transcript(event=event, level=2)
            
            if not transcript or not transcript.speakers:
                continue

            # 3. Apply the split logic to create our lists of dicts
            prepped, qa = split_transcript_by_turns(transcript.speakers)
            
            # Store results with metadata for the training table
            historical_data.append((t_id,
                ticker,
                event.conference_date.strftime('%Y-%m-%d %H:%M:%S'), # Real Timestamp
                effective_date,                                      # Trading Day 0
                f_year,
                f_period,
                json.dumps(prepped),
                json.dumps(qa)))

        except Exception as e:
            print(f"Error processing Q{event.quarter} {event.year}: {e}")
        
        if historical_data:
            
            cursor.executemany("""
                INSERT OR REPLACE INTO transcripts 
                (transcript_id, ticker, event_timestamp, effective_date, fiscal_year, fiscal_period, content_prepped, content_qa) 
                VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """, historical_data)
        print(f"\n✅ Total Ingested: {len(historical_data)} filings.")
        conn.commit()
    return historical_data

def split_transcript_by_turns(turns):
    """
    Processes Level 2 turns into two lists of dictionaries based on the Q&A pivot.
    """
    prepped_list = []
    qa_list = []
    is_qa_started = False
    
    # MSFT specific pivot markers
    qa_markers = ["q and a", "q&a","q and a"]

    for turn in turns:
        speaker_name = turn.speaker_info.name if turn.speaker else "Unknown"
        speaker_title = turn.speaker_info.title if turn.speaker else "Unknown" 
        text = turn.text if turn.text else ""
        text_lower = text.lower()

        # Detect transition
        if not is_qa_started:
            if any(marker in text_lower for marker in qa_markers):
                is_qa_started = True
                # We skip the specific turn that announces the Q&A to keep data clean
                continue 

        # Add to respective list as a dictionary
        entry = {"speaker_name": speaker_name,'speaker_title':speaker_title, "text": text}
        
        if is_qa_started:
            qa_list.append(entry)
        else:
            prepped_list.append(entry)

    return prepped_list, qa_list

# Execute
msft_dataset = process_msft_history(cursor)

Ingesting: Q2 2026 (Date: 2026-01-28)

✅ Total Ingested: 1 filings.
Ingesting: Q1 2026 (Date: 2025-10-29)

✅ Total Ingested: 2 filings.
Ingesting: Q4 2025 (Date: 2025-07-30)

✅ Total Ingested: 3 filings.
Ingesting: Q3 2025 (Date: 2025-04-30)

✅ Total Ingested: 4 filings.
Ingesting: Q2 2025 (Date: 2025-01-29)

✅ Total Ingested: 5 filings.
Ingesting: Q1 2025 (Date: 2024-10-30)

✅ Total Ingested: 6 filings.
Ingesting: Q4 2024 (Date: 2024-07-30)

✅ Total Ingested: 7 filings.
Ingesting: Q3 2024 (Date: 2024-04-25)

✅ Total Ingested: 8 filings.
Ingesting: Q2 2024 (Date: 2024-01-30)

✅ Total Ingested: 9 filings.
Ingesting: Q1 2024 (Date: 2023-10-24)

✅ Total Ingested: 10 filings.
Ingesting: Q4 2023 (Date: 2023-07-25)

✅ Total Ingested: 11 filings.
Ingesting: Q3 2023 (Date: 2023-04-25)

✅ Total Ingested: 12 filings.
Ingesting: Q2 2023 (Date: 2023-01-24)

✅ Total Ingested: 13 filings.
Ingesting: Q1 2023 (Date: 2022-10-25)

✅ Total Ingested: 14 filings.
Ingesting: Q4 2022 (Date: 2022-07-26)

✅ To

In [ ]:
msft_dataset

In [ ]:
def initialize_transcript_tables(client):
    # Table A: Metadata
    client.execute('''
        CREATE TABLE IF NOT EXISTS earnings_calls (
            call_id TEXT PRIMARY KEY,
            ticker TEXT,
            date TEXT,
            year INTEGER,
            quarter INTEGER
        )
    ''')
    
    # Table B: Individual Speech Acts
    client.execute('''
        CREATE TABLE IF NOT EXISTS call_transcript_turns (
            turn_id TEXT PRIMARY KEY,
            call_id TEXT,
            section TEXT,       -- 'PREP' or 'QA'
            sequence INTEGER,   -- The order of speaking
            speaker TEXT,
            content TEXT,
            sentiment_score REAL DEFAULT 0.0
        )
    ''')
    

def sync_transcripts_to_turso(ticker, transcripts_list):
    """
    transcripts_list: Your [dict, dict, ...] from the previous step
    """
    initialize_transcript_tables()
    
    call_rows = []
    turn_rows = []
    
    for call in transcripts_list:
        call_id = f"{ticker}_{call['year']}_Q{call['quarter']}"
        
        # 1. Add to Metadata
        call_rows.append((call_id, ticker, call['date'], call['year'], call['quarter']))
        
        # 2. Add Prepared Remarks (Section: PREP)
        for i, (speaker, text) in enumerate(call['prepped_text']):
            turn_id = f"{call_id}_PREP_{i}"
            turn_rows.append((turn_id, call_id, 'PREP', i, speaker, text, 0.0))
            
        # 3. Add Q&A Section (Section: QA)
        for i, (speaker, text) in enumerate(call['qa']):
            turn_id = f"{call_id}_QA_{i}"
            turn_rows.append((turn_id, call_id, 'QA', i, speaker, text, 0.0))

    # Batch Insert (Production Style)
    cursor.executemany("INSERT OR REPLACE INTO earnings_calls VALUES (?, ?, ?, ?, ?)", call_rows)
    cursor.executemany("INSERT OR REPLACE INTO call_transcript_turns VALUES (?, ?, ?, ?, ?, ?, ?)", turn_rows)
    
    conn.commit()
    print(f"✅ Synced {len(call_rows)} calls and {len(turn_rows)} individual speech turns.")

# EXECUTE
# sync_transcripts_to_turso('MSFT', my_transcripts_list)

In [ ]:
import sys
print(sys.version)

In [ ]:
dat = yf.Ticker("MSFT")


In [ ]:
dat.info()